In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib, sys, subprocess
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# xgboost may not be installed in the kernel; attempt to import
try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Paths
out_dir = Path(r"C:\Users\asus\OneDrive\Desktop\AICTE_EV_PROJECT")
X_train_path = out_dir / 'X_train_preprocessed.csv'
X_test_path = out_dir / 'X_test_preprocessed.csv'
y_train_path = out_dir / 'y_train.csv'
y_test_path = out_dir / 'y_test.csv'

# Load data (preprocessed expected)
if X_train_path.exists() and X_test_path.exists() and y_train_path.exists() and y_test_path.exists():
    X_train = pd.read_csv(X_train_path)
    X_test = pd.read_csv(X_test_path)
    y_train = pd.read_csv(y_train_path).squeeze()
    y_test = pd.read_csv(y_test_path).squeeze()
    print('Loaded preprocessed train/test from CSV')
else:
    raise FileNotFoundError('Preprocessed train/test CSVs not found. Run preprocessing pipeline cell first.')

print('Shapes:', X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# Convert to numpy if needed for XGBoost
X_train_arr = X_train.values if hasattr(X_train, 'values') else X_train
X_test_arr = X_test.values if hasattr(X_test, 'values') else X_test

# Define parameter distributions (kept reasonably small for speed)
rf_param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 6, 12, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}
xgb_param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}
gb_param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 6, 9],
    'subsample': [0.6, 0.8, 1.0],
}

# Models to tune
tune_jobs = [
    ('RandomForest', RandomForestRegressor(random_state=42), rf_param_dist),
    # only include XGBoost if available in this environment
    ('XGBoost', XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0), xgb_param_dist) if XGBRegressor is not None else None,
    ('GradientBoosting', GradientBoostingRegressor(random_state=42), gb_param_dist),
]
# filter out any None entries (when XGBoost missing)
tune_jobs = [t for t in tune_jobs if t is not None]

results = []
cv = KFold(n_splits=3, shuffle=True, random_state=42)
for name, estimator, param_dist in tune_jobs:
    print(f'\nTuning {name} with RandomizedSearchCV...')
    rs = RandomizedSearchCV(estimator=estimator, param_distributions=param_dist, n_iter=20,
                        scoring='neg_mean_squared_error', cv=cv, n_jobs=-1, random_state=42, verbose=1)
    try:
        rs.fit(X_train_arr, y_train)
    except Exception as e:
        print('Tuning failed for', name, '->', e)
        continue
    best = rs.best_estimator_
source
    y_pred = best.predict(X_test_arr)
    test_rmse = mean_squared_error(y_test, y_pred, squared=False)
    test_mae = mean_absolute_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)
    # Save the tuned model
    joblib.dump(best, out_dir / f'model_tuned_{name}.joblib')
    print(f'Saved tuned {name} to', out_dir / f'model_tuned_{name}.joblib')
    # Record CV best score (convert neg_mse -> rmse)
    best_cv_neg_mse = rs.best_score_
    best_cv_rmse = ( -best_cv_neg_mse ) ** 0.5 if best_cv_neg_mse is not None else None
    results.append({
        'model': name,
        'best_params': rs.best_params_,
        'cv_rmse': best_cv_rmse,
        'test_rmse': float(test_rmse),
        'test_mae': float(test_mae),
        'test_r2': float(test_r2),
    })

# Save results summary
res_df = pd.DataFrame(results)
res_df.to_csv(out_dir / 'tuned_models_results.csv', index=False)
print('Tuning complete. Results saved to', out_dir / 'tuned_models_results.csv')
display(res_df)


Columns after cleaning: ['brand', 'model', 'top_speed_kmh', 'battery_capacity_kwh', 'battery_type', 'number_of_cells', 'torque_nm', 'efficiency_wh_per_km', 'range_km', 'acceleration_0_100_s', 'fast_charging_power_kw_dc', 'fast_charge_port', 'towing_capacity_kg', 'cargo_volume_l', 'seats', 'drivetrain', 'segment', 'length_mm', 'width_mm', 'height_mm', 'car_body_type', 'source_url']
Converted column to numeric: cargo_volume_l
Numeric columns: ['top_speed_kmh', 'battery_capacity_kwh', 'number_of_cells', 'torque_nm', 'efficiency_wh_per_km', 'range_km', 'acceleration_0_100_s', 'fast_charging_power_kw_dc', 'towing_capacity_kg', 'cargo_volume_l', 'seats', 'length_mm', 'width_mm', 'height_mm']
Categorical columns (first 20): ['brand', 'model', 'battery_type', 'fast_charge_port', 'drivetrain', 'segment', 'car_body_type', 'source_url']
No missing values remain after fills
Saved cleaned CSV to C:\Users\asus\OneDrive\Desktop\AICTE_EV_PROJECT\electric_vehicles_spec_2025_cleaned.csv
Shape before -> 

C:\Users\asus\AppData\Local\Temp\ipykernel_20320\2530189292.py:56: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[c].fillna(med, inplace=True)
C:\Users\asus\AppData\Local\Temp\ipykernel_20320\2530189292.py:56: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 

,brand,model,top_speed_kmh,battery_capacity_kwh,battery_type,number_of_cells,torque_nm,efficiency_wh_per_km,range_km,acceleration_0_100_s,...,towing_capacity_kg,cargo_volume_l,seats,drivetrain,segment,length_mm,width_mm,height_mm,car_body_type,source_url
0,Abarth,500e Convertible,155,37.8,Lithium-ion,192.0,235.0,156,225,7.0,...,0.0,185.0,4,FWD,B - Compact,3673,1683,1518,Hatchback,https://ev-database.org/car/1904/Abarth-500e-C...
1,Abarth,500e Hatchback,155,37.8,Lithium-ion,192.0,235.0,149,225,7.0,...,0.0,185.0,4,FWD,B - Compact,3673,1683,1518,Hatchback,https://ev-database.org/car/1903/Abarth-500e-H...
2,Abarth,600e Scorpionissima,200,50.8,Lithium-ion,102.0,345.0,158,280,5.9,...,0.0,360.0,5,FWD,JB - Compact,4187,1779,1557,SUV,https://ev-database.org/car/3057/Abarth-600e-S...
3,Abarth,600e Turismo,200,50.8,Lithium-ion,102.0,345.0,158,280,6.2,...,0.0,360.0,5,FWD,JB - Compact,4187,1779,1557,SUV,https://ev-database.org/car/3056/Abarth-600e-T...
4,Aiways,U5,150,60.0,Lithium-ion,216.0,310.0,156,315,7.5,...,1000.0,496.0,5,FWD,JC - Medium,4680,1865,1700,SUV,https://ev-database.org/car/1678/Aiways-U5



Column dtypes:
brand                         object
model                         object
top_speed_kmh                  int64
battery_capacity_kwh         float64
battery_type                  object
number_of_cells              float64
torque_nm                    float64
efficiency_wh_per_km           int64
range_km                       int64
acceleration_0_100_s         float64
fast_charging_power_kw_dc    float64
fast_charge_port              object
towing_capacity_kg           float64
cargo_volume_l               float64
seats                          int64
drivetrain                    object
segment                       object
length_mm                      int64
width_mm                       int64
height_mm                      int64
car_body_type                 object
source_url                    object
dtype: object

Value counts (sample) for first categorical columns:

-- brand 
 brand
Mercedes-Benz    42
Audi             28
Porsche          26
Volkswagen       23
Ford

## Feature engineering: scaling & one-hot encoding
This cell will:
- load the cleaned CSV
- detect numeric and categorical columns
- apply StandardScaler and MinMaxScaler to numeric columns
- one-hot encode low-cardinality categorical columns (cardinality < 30)
- drop/ignore very high-cardinality categorical columns (printed)
- save two feature matrices: `features_standard_scaled.csv` and `features_minmax_scaled.csv`
- save scaler objects as joblib files for reuse

In [ ]:
from pathlib import Path
import sys, subprocess
import pandas as pd
import numpy as np

# Ensure scikit-learn and joblib available in this kernel; install if missing
try:
    from sklearn.preprocessing import StandardScaler, MinMaxScaler
    import joblib
except Exception as e:
    print('scikit-learn or joblib not present; installing scikit-learn and joblib...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn', 'joblib'])
    from sklearn.preprocessing import StandardScaler, MinMaxScaler
    import joblib

# Paths
clean_path = Path(r"C:\Users\asus\OneDrive\Desktop\AICTE_EV_PROJECT\electric_vehicles_spec_2025_cleaned.csv")
out_dir = clean_path.parent

# Load cleaned dataframe
df = pd.read_csv(clean_path)
print('Loaded cleaned df with shape', df.shape)

# Identify numeric and categorical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print('Numeric columns (count):', len(num_cols))
print('Categorical columns (count):', len(cat_cols))

# Choose categorical columns to one-hot encode: low cardinality (<30)
low_card_cols = [c for c in cat_cols if df[c].nunique() < 30]
high_card_cols = [c for c in cat_cols if df[c].nunique() >= 30]
print('One-hot encoding these categorical cols:', low_card_cols)
if high_card_cols:
    print('High-cardinality categorical columns (excluded from one-hot):', high_card_cols)

# Build dummies for low-cardinality categoricals
if low_card_cols:
    dummies = pd.get_dummies(df[low_card_cols].astype(str), prefix=low_card_cols, drop_first=False)
else:
    dummies = pd.DataFrame(index=df.index)

# Numeric features to scale
X_num = df[num_cols].copy()
# If any numeric columns are integer typed but represent categories (like seats), we still scale them — user can adjust later
print('Numeric feature columns:', num_cols)

# Fit Standard scaler and MinMax scaler on numeric features
    print(f"{name} CV R2 mean/std: {np.nanmean(r2):.3f} ± {np.nanstd(r2):.3f}")
    print(f"{name} CV RMSE mean/std: {np.nanmean(rmse):.3f} ± {np.nanstd(rmse):.3f}")
    print(f"{name} Test RMSE: {test_rmse:.3f} MAE: {test_mae:.3f} R2: {test_r2:.3f}")
X_mm_num = pd.DataFrame(scaler_mm.fit_transform(X_num), columns=num_cols)

# Combine scaled numeric features with dummies
X_std = pd.concat([X_std_num.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
X_mm = pd.concat([X_mm_num.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

# Save feature matrices and scalers
std_out = out_dir / 'features_standard_scaled.csv'
mm_out = out_dir / 'features_minmax_scaled.csv'
scaler_std_out = out_dir / 'scaler_standard.joblib'
scaler_mm_out = out_dir / 'scaler_minmax.joblib'
X_std.to_csv(std_out, index=False)
X_mm.to_csv(mm_out, index=False)
joblib.dump(scaler_std, scaler_std_out)
res_df.to_csv(out_dir / 'model_comparison_results.csv', index=False)
print('Model comparison saved to', out_dir / 'model_comparison_results.csv')
display(res_df)

print('Saved scalers:')
print(' -', scaler_std_out)
print(' -', scaler_mm_out)

# Quick sanity checks
print('\nSample of feature matrix (standard scaled):')
display(X_std.head())
print('\nColumn count:', X_std.shape[1])

scikit-learn or joblib not present; installing scikit-learn and joblib...
Loaded cleaned df with shape (478, 22)
Numeric columns (count): 14
Categorical columns (count): 8
One-hot encoding these categorical cols: ['battery_type', 'fast_charge_port', 'drivetrain', 'segment', 'car_body_type']
High-cardinality categorical columns (excluded from one-hot): ['brand', 'model', 'source_url']
Numeric feature columns: ['top_speed_kmh', 'battery_capacity_kwh', 'number_of_cells', 'torque_nm', 'efficiency_wh_per_km', 'range_km', 'acceleration_0_100_s', 'fast_charging_power_kw_dc', 'towing_capacity_kg', 'cargo_volume_l', 'seats', 'length_mm', 'width_mm', 'height_mm']
Saved feature matrices:
 - C:\Users\asus\OneDrive\Desktop\AICTE_EV_PROJECT\features_standard_scaled.csv shape (478, 43)
 - C:\Users\asus\OneDrive\Desktop\AICTE_EV_PROJECT\features_minmax_scaled.csv shape (478, 43)
Saved scalers:
 - C:\Users\asus\OneDrive\Desktop\AICTE_EV_PROJECT\scaler_standard.joblib
 - C:\Users\asus\OneDrive\Desktop\A

,top_speed_kmh,battery_capacity_kwh,number_of_cells,torque_nm,efficiency_wh_per_km,range_km,acceleration_0_100_s,fast_charging_power_kw_dc,towing_capacity_kg,cargo_volume_l,...,segment_JF - Luxury,segment_N - Passenger Van,car_body_type_Cabriolet,car_body_type_Coupe,car_body_type_Hatchback,car_body_type_Liftback Sedan,car_body_type_SUV,car_body_type_Sedan,car_body_type_Small Passenger Van,car_body_type_Station/Estate
0,-0.891005,-1.784545,-0.193421,-1.093690,-0.201384,-1.629978,0.043025,-0.998236,-1.464016,-1.659132,...,False,False,False,False,True,False,False,False,False,False
1,-0.891005,-1.784545,-0.193421,-1.093690,-0.405575,-1.629978,0.043025,-0.998236,-1.464016,-1.659132,...,False,False,False,False,True,False,False,False,False,False
2,0.424134,-1.144460,-0.290406,-0.634536,-0.143044,-1.096925,-0.360225,-0.791645,-1.464016,-0.718462,...,False,False,False,False,False,False,True,False,False,False
3,0.424134,-1.144460,-0.290406,-0.634536,-0.143044,-1.096925,-0.250248,-0.791645,-1.464016,-0.718462,...,False,False,False,False,False,False,True,False,False,False
4,-1.037131,-0.691476,-0.167559,-0.780631,-0.201384,-0.757710,0.226320,-0.808861,-0.068942,0.012572,...,False,False,False,False,False,False,True,False,False,False



Column count: 43


## ML-ready preprocessing pipeline
This cell builds a reusable sklearn ColumnTransformer pipeline that:
- selects a target (assumes `range_km` by default)
- frequency-encodes high-cardinality categoricals (brand/model/source_url)
- one-hot-encodes low-cardinality categoricals
- imputes and standard-scales numeric features
- splits into train/test and saves preprocessed feature CSVs and the pipeline/joblib

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, joblib, sys, subprocess
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

# Ensure sklearn is available
try:
    import sklearn
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn'])
    import sklearn

# Paths
clean_path = Path(r"C:\Users\asus\OneDrive\Desktop\AICTE_EV_PROJECT\electric_vehicles_spec_2025_cleaned.csv")
out_dir = clean_path.parent

# Load cleaned dataframe
df = pd.read_csv(clean_path)
print('Loaded cleaned df with shape', df.shape)

# Choose target column (assumption). If not present, pick the first numeric column and warn
target = 'range_km'
if target not in df.columns:
    num_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_candidates:
        target = num_candidates[0]
        print(f'Warning: default target "range_km" not found. Using {target} as target.')
    else:
        raise ValueError('No numeric column found to use as target; please specify a target column')

y = df[target].copy()
X = df.drop(columns=[target]).copy()

# Detect categorical and numeric features
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
print('Initial numeric cols:', num_cols)
print('Initial categorical cols:', cat_cols)

# High-cardinality handling: frequency encode columns with nunique >= 30
high_card_cols = [c for c in cat_cols if X[c].nunique() >= 30]
low_card_cols = [c for c in cat_cols if X[c].nunique() < 30]
print('High-cardinality cols (freq-encoded):', high_card_cols)
print('Low-cardinality cols (one-hot):', low_card_cols)

# Create frequency columns for high-cardinality categoricals
for c in high_card_cols:
    freq = X[c].value_counts(normalize=True)
    X[c + '_freq'] = X[c].map(freq).fillna(0)

freq_cols = [c + '_freq' for c in high_card_cols]
# Add freq cols to numeric list for scaling
num_cols_extended = num_cols + freq_cols

# Build pipelines
num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols_extended),
    ('cat', cat_pipeline, low_card_cols)
], remainder='drop')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit preprocessor on training data and transform
preprocessor.fit(X_train)
X_train_p = preprocessor.transform(X_train)
X_test_p = preprocessor.transform(X_test)

# Get feature names (scikit-learn >=1.0 provides get_feature_names_out)
try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    # Fallback: build names manually
    num_feat_names = num_cols_extended
    if low_card_cols:
        ohe = preprocessor.named_transformers_['cat'].named_steps['ohe']
        ohe_names = ohe.get_feature_names_out(low_card_cols).tolist()
    else:
        ohe_names = []
    feature_names = list(num_feat_names) + list(ohe_names)

# Build DataFrames from transformed arrays
X_train_df = pd.DataFrame(X_train_p, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_p, columns=feature_names, index=X_test.index)

# Save outputs and pipeline
X_train_df.to_csv(out_dir / 'X_train_preprocessed.csv', index=False)
X_test_df.to_csv(out_dir / 'X_test_preprocessed.csv', index=False)
y_train.to_csv(out_dir / 'y_train.csv', index=False)
y_test.to_csv(out_dir / 'y_test.csv', index=False)
joblib.dump(preprocessor, out_dir / 'preprocessor_pipeline.joblib')
print('Saved preprocessed train/test and pipeline in', out_dir)
print('X_train shape:', X_train_df.shape, 'X_test shape:', X_test_df.shape)
print('Target column used:', target)


## Data quality & baseline diagnostics
This cell runs quick diagnostics to help decide model suitability:
- target distribution and basic stats
- correlation of numeric features with target
- pairwise high correlations (multicollinearity check)
- baseline CV results for LinearRegression and RandomForestRegressor

In [19]:
from pathlib import Path
import pandas as pd, numpy as np, joblib, sys, subprocess
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load cleaned data
clean_path = Path(r"C:\Users\asus\OneDrive\Desktop\AICTE_EV_PROJECT\electric_vehicles_spec_2025_cleaned.csv")
df = pd.read_csv(clean_path)
print('Loaded cleaned df, shape:', df.shape)

# Target
target = 'range_km'
if target not in df.columns:
    raise ValueError('Expected target column `range_km` not found in cleaned data')
y = df[target]
X = df.drop(columns=[target])

# Target stats
print('\nTarget (`range_km`) stats:')
print(' count', y.count())
print(' mean', round(y.mean(),2))
print(' median', round(y.median(),2))
print(' std', round(y.std(),2))
print(' min', y.min(), 'max', y.max())
print(' 25%', y.quantile(0.25), ' 75%', y.quantile(0.75))

# Numeric / categorical columns
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print('\nNumeric cols:', num_cols)
print('Categorical cols:', cat_cols)

# Correlation of numeric features with target
corrs = df[num_cols + [target]].corr()[target].drop(target).sort_values(ascending=False)
print('\nTop correlations with target:')
print(corrs.head(10))

# Pairwise high correlations among numeric features (abs>0.9)
corr_mat = df[num_cols].corr().abs()
high_pairs = []
for i in range(len(num_cols)):
    for j in range(i+1, len(num_cols)):
        a = num_cols[i]; b = num_cols[j]
        if corr_mat.loc[a,b] > 0.9:
            high_pairs.append((a,b,corr_mat.loc[a,b]))
if high_pairs:
    print('\nHighly correlated numeric pairs (abs corr > 0.9):')
    for p in high_pairs:
        print(p)
else:
    print('\nNo extremely high (>0.9) pairwise correlations among numeric features')
# Prepare features using preprocessor if available, else do a simple numeric+ohe transform
preproc_path = clean_path.parent / 'preprocessor_pipeline.joblib'
if preproc_path.exists():
    preprocessor = joblib.load(preproc_path)
    X_all = preprocessor.transform(X)
    try:
        feat_names = preprocessor.get_feature_names_out()
    except Exception:
        feat_names = None
else:
    # fallback: numeric only
    print('\nPreprocessor not found; using numeric columns only for baseline models')
    X_all = X[num_cols].values
    feat_names = num_cols

# Baseline models: LinearRegression and RandomForest (5-fold CV)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
models = {'LinearRegression': LinearRegression(), 'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42)}
results = {}
for name, model in models.items():
    print(f'\nRunning CV for {name}...')
    # R2 scores
    r2 = cross_val_score(model, X_all, y, cv=kf, scoring='r2')
    # RMSE (neg MSE -> sqrt)
    neg_mse = cross_val_score(model, X_all, y, cv=kf, scoring='neg_mean_squared_error')
    rmse = np.sqrt(-neg_mse)
    print(f' {name} R2 mean/std: {r2.mean():.3f} ± {r2.std():.3f}')
    print(f' {name} RMSE mean/std: {rmse.mean():.3f} ± {rmse.std():.3f}')
    results[name] = {'r2_mean': r2.mean(), 'r2_std': r2.std(), 'rmse_mean': rmse.mean(), 'rmse_std': rmse.std()}

# Feature importance from RandomForest trained on full data (if names available)
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_all, y)
if feat_names is not None:
    try:
        imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)
        print('\nTop 10 feature importances (RandomForest):')
        print(imp.head(10))
    except Exception:
        pass

print('\nDone diagnostics')

Loaded cleaned df, shape: (478, 22)

Target (`range_km`) stats:
 count 478
 mean 393.18
 median 397.5
 std 103.29
 min 135 max 685
 25% 320.0  75% 470.0

Numeric cols: ['top_speed_kmh', 'battery_capacity_kwh', 'number_of_cells', 'torque_nm', 'efficiency_wh_per_km', 'acceleration_0_100_s', 'fast_charging_power_kw_dc', 'towing_capacity_kg', 'cargo_volume_l', 'seats', 'length_mm', 'width_mm', 'height_mm']
Categorical cols: ['brand', 'model', 'battery_type', 'fast_charge_port', 'drivetrain', 'segment', 'car_body_type', 'source_url']

Top correlations with target:
battery_capacity_kwh         0.880433
top_speed_kmh                0.732130
fast_charging_power_kw_dc    0.720123
torque_nm                    0.636849
width_mm                     0.521392
length_mm                    0.496867
towing_capacity_kg           0.329475
number_of_cells              0.237162
efficiency_wh_per_km         0.022943
cargo_volume_l              -0.065286
Name: range_km, dtype: float64

No extremely high (>0.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, joblib, sys
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Optional XGBoost: include only if available in the environment (do NOT auto-install here)
try:
    from xgboost import XGBRegressor
    has_xgb = True
except Exception:
    XGBRegressor = None
    has_xgb = False

out_dir = Path(r"C:\Users\asus\OneDrive\Desktop\AICTE_EV_PROJECT")
X_train_path = out_dir / 'X_train_preprocessed.csv'
X_test_path = out_dir / 'X_test_preprocessed.csv'
y_train_path = out_dir / 'y_train.csv'
y_test_path = out_dir / 'y_test.csv'

# Load preprocessed train/test if present, else fail and ask user to run the preprocessing cell
if X_train_path.exists() and X_test_path.exists() and y_train_path.exists() and y_test_path.exists():
    X_train = pd.read_csv(X_train_path)
    X_test = pd.read_csv(X_test_path)
    y_train = pd.read_csv(y_train_path).squeeze()
    y_test = pd.read_csv(y_test_path).squeeze()
    print('Loaded preprocessed train/test from CSV')
else:
    raise FileNotFoundError('Preprocessed train/test CSVs not found. Run the preprocessing pipeline cell first.')

print('Shapes:', X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# Ensure models dict includes XGBoost only when available
models = {
    'LinearRegression': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, random_state=42),
    'SVR': SVR(),
    'KNN': KNeighborsRegressor(n_neighbors=5)
}
if has_xgb:
    models['XGBoost'] = XGBRegressor(n_estimators=200, random_state=42, objective='reg:squarederror')

results = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f'\nTraining & CV for {name}...')
    # cross-val on training set
    try:
        r2 = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2', n_jobs=-1)
        neg_mse = cross_val_score(model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
        rmse = np.sqrt(-neg_mse)
    except Exception as e:
        print('CV failed for', name, ':', e)
        r2 = np.array([np.nan])
        rmse = np.array([np.nan])
    # fit on full train and evaluate on holdout
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        test_mae = mean_absolute_error(y_test, y_pred)
        test_r2 = r2_score(y_test, y_pred)
    except Exception as e:
        print('Train/eval failed for', name, ':', e)
        test_rmse = np.nan; test_mae = np.nan; test_r2 = np.nan
    # save fitted model
    joblib.dump(model, out_dir / f'model_{name}.joblib')
    print(f'Saved model_{name}.joblib')
    results.append({
        'model': name,
        'cv_r2_mean': float(np.nanmean(r2)),
        'cv_r2_std': float(np.nanstd(r2)),
        'cv_rmse_mean': float(np.nanmean(rmse)),
        'cv_rmse_std': float(np.nanstd(rmse)),
        'test_rmse': float(test_rmse) if not np.isnan(test_rmse) else None,
        'test_mae': float(test_mae) if not np.isnan(test_mae) else None,
        'test_r2': float(test_r2) if not np.isnan(test_r2) else None
    })

res_df = pd.DataFrame(results).sort_values('test_rmse')
res_df.to_csv(out_dir / 'model_comparison_results.csv', index=False)
print('Model comparison saved to', out_dir / 'model_comparison_results.csv')
display(res_df)
